# **Phase 5: Decision Framework Comparison**
---
Compares three independent technology-selection frameworks: cluster-only,
rule-only (process suitability), and the hybrid (cluster + rule) framework
from Phase 4C.

In [ ]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.models.cluster_only_rules import assign_cluster_only_decision   # cluster only
from src.models.suitability_rules import apply_process_rules            # pri/sec/constraint
from src.models.rule_only_conversion import assign_rule_only_conversion     # rule only
from src.models.wte_conversion_rules import assign_conversion_technology    # hybrid logic

# Load data
df = pd.read_csv("../data/interim/engineered_features_with_cluster.csv")

df = assign_cluster_only_decision(df)       # Cluster-only decision
df = apply_process_rules(df)                # Process suitability/feasibility rules
df = assign_rule_only_conversion(df)        # Rule-only decision
df = assign_conversion_technology(df)       # Hybrid decision

df.to_csv(
    "../data/processed/comparison_decisions.csv",
    index=False
)

In [2]:
df

,Sample_ID,Biomass_Type,Class,Subclass,Ash_db,VM_db,FC_db,C_db,H_db,N_db,...,Moist_ar,Moisture_Penalty,Effective_HHV,Cluster,Final_Tech_ClusterOnly,Primary_Process,Secondary_Process,Constraint_Level,Final_Tech_RuleOnly,Final_Conversion_Technology
0,1,Industrial Processing,Timber industry,Woodchips (Softwood),1.10,81.00,17.90,48.60,6.26,0.20,...,41.2,0.023697,-826.110,1.0,Pyrolysis,Pyrolysis,NaN,High,Pyrolysis,Pyrolysis
1,3,Agricultural,Animal farming,Chicken manure pellets,32.70,56.20,11.10,30.00,3.91,3.94,...,18.9,0.050251,-222.139,2.0,Further Assessment Needed,Pre-treatment Required,NaN,Low,Further Assessment Needed,Pyrolysis
2,4,Urban Waste,Biosolids,Treated biosolids,42.80,52.00,5.20,24.70,4.35,4.61,...,8.1,0.109890,-89.886,2.0,Further Assessment Needed,Pre-treatment Required,NaN,High,Further Assessment Needed,Gasification
3,5,Industrial Processing,Paper industry,Paper sludge,26.20,64.20,9.60,32.40,4.96,0.47,...,7.8,0.113636,-96.152,2.0,Further Assessment Needed,Pre-treatment Required,NaN,Low,Further Assessment Needed,Pyrolysis
4,6,Industrial Processing,Cotton Industry,Cotton seed hulls,1.90,77.90,20.20,32.50,6.02,0.45,...,11.6,0.079365,-193.768,1.0,Pyrolysis,Pyrolysis,NaN,Moderate,Pyrolysis,Pyrolysis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,138,Industrial Processing,Other,Hardwood (Eucalyptus),8.00,73.40,18.60,49.00,5.91,0.13,...,6.6,0.131579,-111.496,1.0,Pyrolysis,Pyrolysis,NaN,Moderate,Pyrolysis,Pyrolysis
131,139,Industrial Processing,Other,Wood waste and plastics waste,8.41,71.05,20.54,45.49,5.68,1.27,...,40.3,0.024213,-730.194,1.0,Pyrolysis,Pyrolysis,NaN,High,Pyrolysis,Pyrolysis
132,140,Industrial Processing,Other,Wood waste and plastics waste,13.59,68.85,17.56,44.65,5.62,1.05,...,32.5,0.029851,-563.850,1.0,Pyrolysis,Pyrolysis,NaN,High,Pyrolysis,Pyrolysis
133,141,Agricultural,Other,Algae (Phyllospora),22.69,69.16,8.15,36.10,4.40,0.98,...,NaN,NaN,NaN,NaN,Further Assessment Needed,Pyrolysis,NaN,Moderate,Pyrolysis,Further Assessment Needed


In [3]:
# Comparative summary
comparison = (
    df.groupby("Cluster")[
        [
            "Final_Tech_ClusterOnly",
            "Final_Tech_RuleOnly",
            "Final_Conversion_Technology"
        ]
    ]
    .value_counts()
    .unstack(fill_value=0)
)

comparison.to_csv(
    "../results/tables/phase5_comparison/comparison_summary.csv"
)

_**For each (Cluster × Cluster-only decision × Rule-only decision) combination,
how many samples end up with each final conversion technology?**_


In [4]:
comparison

Final_Conversion_Technology                                  Combustion  \
Cluster Final_Tech_ClusterOnly    Final_Tech_RuleOnly                     
0.0     Combustion                Pyrolysis                           1   
1.0     Pyrolysis                 Pyrolysis                           0   
2.0     Further Assessment Needed Further Assessment Needed           0   
                                  Gasification                        0   
                                  Pyrolysis                           0   

Final_Conversion_Technology                                  Pyrolysis  \
Cluster Final_Tech_ClusterOnly    Final_Tech_RuleOnly                    
0.0     Combustion                Pyrolysis                          0   
1.0     Pyrolysis                 Pyrolysis                        107   
2.0     Further Assessment Needed Further Assessment Needed          2   
                                  Gasification                       1   
                                  Pyrolysis                          1   

Final_Conversion_Technology                                  Gasification  
Cluster Final_Tech_ClusterOnly    Final_Tech_RuleOnly                      
0.0     Combustion                Pyrolysis                             0  
1.0     Pyrolysis                 Pyrolysis                             0  
2.0     Further Assessment Needed Further Assessment Needed             6  
                                  Gasification                          0  
                                  Pyrolysis                             0

_**How often the three decision frameworks agree, to see whether the
hybrid model adds value beyond rules or clusters alone.**_


In [5]:
# valuation tables
# 1. Crosstab: Cluster vs Technology (Hybrid)
hybrid = pd.crosstab(
    df["Cluster"],
    df["Final_Conversion_Technology"]
)

hybrid.to_csv(
    "../results/tables/phase5_comparison/cluster_vs_hybrid.csv"
)

# 2. Agreement comparison
agree_comp = pd.DataFrame({
    "Cluster_vs_Rule_Agreement":
        (df["Final_Tech_ClusterOnly"] == df["Final_Tech_RuleOnly"]).mean(),

    "Rule_vs_Hybrid_Agreement":
        (df["Final_Tech_RuleOnly"] == df["Final_Conversion_Technology"]).mean(),

    "Cluster_vs_Hybrid_Agreement":
        (df["Final_Tech_ClusterOnly"] == df["Final_Conversion_Technology"]).mean(),
}, index=["Agreement_Rate"])

agree_comp.to_csv(
    "../results/tables/phase5_comparison/decision_comparison_agreement_rate.csv"
)

print(hybrid)
print(agree_comp)

Final_Conversion_Technology  Combustion  Gasification  Pyrolysis
Cluster                                                         
0.0                                   1             0          0
1.0                                   0             0        107
2.0                                   0             6          4
                Cluster_vs_Rule_Agreement  Rule_vs_Hybrid_Agreement  \
Agreement_Rate                   0.903704                  0.851852   

                Cluster_vs_Hybrid_Agreement  
Agreement_Rate                     0.925926  


_**Which information source dominates the hybrid framework, i.e whether hybrid decisions merely reproduce rule-based logic, merely reproduce unsupervised structure, or meaningfully integrate both, quantified via agreement rates between decision layers?**_

**Interpretation:** All three frameworks show high mutual agreement (90–93%)
on the dominant fuel population (the large, moderate-ash, high-volatile-matter
majority cluster suited to pyrolysis). 
Disagreement concentrates almost entirely in the small, high-ash Cluster 2 subgroup and in the handful of
samples lacking a fuel-typology cluster (see Phase 6A for a targeted
breakdown of these disagreement cases).

| Comparison | Agreement Rate | Disagreement Rate | Interpretation |
|---|---|---|---|
| Cluster vs Rule | 90.4% | 9.6% | Fuel typology and engineering thresholds largely agree, since both correctly identify the dominant moderate-ash / high-volatile fuel population as pyrolysis-suited |
| Rule vs Hybrid | 85.2% | 14.8% | The hybrid framework mostly follows rule-only logic, but overrides it for high-ash Cluster 2 fuels (using constraint level to choose between gasification and pyrolysis) and for un-clustered samples (deferring to further assessment) |
| Cluster vs Hybrid | 92.6% | 7.4% | Hybrid decisions closely track fuel-typology clusters, since cluster identity anchors the technology choice for the two well-populated clusters |

### **Visualization: Decision Comparison**
---

1. **Bar plots** -> How conservative or aggressive each framework is
2. **Heatmaps** -> Whether clusters dominate decisions or rules override them
3. **Agreement bar** -> Where the hybrid framework actually adds information
4. **Decision flow** -> Stability vs. volatility of decisions across frameworks


In [6]:
# import plots functions
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.visualization.decision_plots import (
    plot_technology_counts,
    plot_cluster_technology_heatmap,
    plot_agreement_rates,
    plot_decision_flow
)

In [7]:
# load phase 5 results
df_vis = pd.read_csv(
    "../data/processed/comparison_decisions.csv"
)

agreement_df = pd.read_csv(
    "../results/tables/phase5_comparison/decision_comparison_agreement_rate.csv",
    index_col=0
)

In [8]:
# 1. technology distribution
# Cluster-only
plot_technology_counts(
    df=df_vis,
    column="Final_Tech_ClusterOnly",
    title="Cluster-only Technology Distribution",
    output_path="../results/figures/phase5_comparison/bar/cluster_only_counts.png"
)

# Rule-only
plot_technology_counts(
    df=df_vis,
    column="Final_Tech_RuleOnly",
    title="Rule-only Technology Distribution",
    output_path="../results/figures/phase5_comparison/bar/rule_only_counts.png"
)

# Hybrid (cluster+rule)
plot_technology_counts(
    df=df_vis,
    column="Final_Conversion_Technology",
    title="Hybrid Technology Distribution",
    output_path="../results/figures/phase5_comparison/bar/hybrid_counts.png"
)


In [9]:
# Cluster vs technology(heatmaps)
# cluster only
plot_cluster_technology_heatmap(
    df=df_vis,
    tech_column="Final_Tech_ClusterOnly",
    output_path="../results/figures/phase5_comparison/heatmap/cluster_vs_cluster_only.png"
)

# rule only
plot_cluster_technology_heatmap(
    df=df_vis,
    tech_column="Final_Tech_RuleOnly",
    output_path="../results/figures/phase5_comparison/heatmap/cluster_vs_rule_only.png"
)

# hybrid
plot_cluster_technology_heatmap(
    df=df_vis,
    tech_column="Final_Conversion_Technology",
    output_path="../results/figures/phase5_comparison/heatmap/cluster_vs_hybrid.png"
)

In [10]:
# 3. Agreement comparison
plot_agreement_rates(
    agreement_df,
    output_path="../results/figures/phase5_comparison/agreement/agreement_rates.png"
)

In [11]:
# 4. Decision flow across frameworks
plot_decision_flow(
    df=df_vis,
    output_path="../results/figures/phase5_comparison/sankey/decision_flow.png"
)